# **Data Preprocessing –  IC50 Dataset (ChEMBL) of multiple AD targets**

---


## **Objective**

This notebook performs systematic cleaning and preprocessing of the raw ChEMBL activity dataset for the 5-HT6 receptor. The goal is to generate a high-quality, machine-learning–ready dataset for QSAR modeling and pIC50 prediction.


The workflow ensures biological consistency, assay reliability, unit harmonization, and removal of noisy or ambiguous measurements before model development.


---



##**Data Source**

Raw ChEMBL activity dataset containing IC50 measurements for different AD targets.



---



## **Preprocessing Workflow**
###**Step 1 – Load Raw Dataset**

- Import the original CSV file containing activity records.

### **Step 2 – Inspect Dataset Structure**

- Print all column names to understand available fields and metadata.

### **Step 3 – Remove Incomplete Records**

Drop rows with missing values in essential activity-related columns:

- standard_value

- standard_units

- standard_type

- assay_type

- standard_relation

This ensures only complete activity entries are retained.

### **Step 4 – Keep Exact Activity Values**

- Retain only entries where: standard_relation == '='

- This removes ambiguous activity values such as “>”, “<”, etc.

### **Step 5 – Keep Only IC50 Measurements**

- Filter rows where standard_type corresponds to IC50 (case-insensitive).

- This ensures consistency in biological endpoint.

### **Step 6 – Restrict to Relevant Assay Types**

- Keep only binding and functional assays: assay_type ∈ {B, F}

- This improves biological relevance and assay comparability.

### **Step 7 – Filter for Human Targets**

- Retain only entries where: target_organism == "Homo sapiens"

- This ensures species consistency for modeling.

### **Step 8 – Normalize Activity Units**

- Convert activity values to nanomolar (nM):

  -  If unit is µM → multiply by 1000

  - If unit is nM → keep as is

  - Other units → discard

- This standardizes all activity values.

### **Step 9 – Remove Invalid Activity Values**

- Drop rows where IC50 values are missing after conversion.

### **Step 10 – Remove Out-of-Bound IC50 Values**

- Retain only biologically meaningful IC50 values: 0.1 nM ≤ IC50 ≤ 100,000 nM

- Rationale: Extremely low values may indicate experimental or unit errors.

- Very high values (>100 µM) typically represent weak or inactive compounds that add noise to regression models.

### **Step 11 – Calculate pIC50**

- Convert IC50 values (in molar units) to pIC50 using:

     pIC50 = −log10(IC50 in molar)

- This transformation: Improves numerical stability

- Normalizes scale: Produces a regression-friendly target variable

### **Step 12 – Final Unit Standardization**

- Retain only entries originally reported in nM to ensure consistency.

### **Step 13 – Reset Index**

- Reorganize dataset indexing after filtering.

### **Step 14 – Save Cleaned Dataset**

- Export the curated dataset for downstream modeling.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Step 1: Load the raw ChEMBL activity dataset
file_path = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/raw-original-data/raw_5-HT6_ic50.csv"
df = pd.read_csv(file_path)

In [ ]:
df.head()

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,NaN,NaN,103650,[],CHEMBL620798,Antagonistic activity towards 5-hydroxytryptam...,F,NaN,NaN,BAO_0000190,...,Homo sapiens,Serotonin 6 (5-HT6) receptor,9606,NaN,NaN,IC50,uM,UO_0000065,NaN,0.8
1,NaN,NaN,103652,[],CHEMBL620798,Antagonistic activity towards 5-hydroxytryptam...,F,NaN,NaN,BAO_0000190,...,Homo sapiens,Serotonin 6 (5-HT6) receptor,9606,NaN,NaN,IC50,uM,UO_0000065,NaN,0.3
2,NaN,NaN,884696,[],CHEMBL618092,Inhibition against human 5-hydroxytryptamine 6...,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Serotonin 6 (5-HT6) receptor,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,0.7
3,NaN,NaN,949636,[],CHEMBL618090,Inhibition of human 5-hydroxytryptamine 6 rece...,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Serotonin 6 (5-HT6) receptor,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,1350.0
4,NaN,NaN,949659,[],CHEMBL618090,Inhibition of human 5-hydroxytryptamine 6 rece...,B,NaN,NaN,BAO_0000190,...,Homo sapiens,Serotonin 6 (5-HT6) receptor,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,427.0


In [ ]:
# Step 2: Print all column names to understand available data
print("All column names in the dataset:")
print(df.columns.tolist())

All column names in the dataset:
['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value']


In [ ]:
df.shape

(1832, 46)

In [ ]:
# Step 3: Drop rows with missing values in key columns required for IC50 prediction
# These include activity values, assay type, target organism, and measurement units
df_clean = df.dropna(subset=["standard_value", "standard_units", "standard_type", "assay_type", "standard_relation"])
print(f"After removing rows with missing values: {df_clean.shape[0]}")

After removing rows with missing values: 1007


In [ ]:
# Step 4: Keep only entries with IC50 values expressed as '='
df_clean = df_clean[df_clean['standard_relation'] == '=']
print(f"After keeping '=' relation entries: {df_clean.shape[0]} rows")

After keeping '=' relation entries: 942 rows


In [ ]:
# Step 5: Retain only IC50 activity type (could be uppercase or lowercase)
df_clean = df_clean[df_clean['standard_type'].str.upper() == 'IC50']

In [ ]:
df_clean.shape

(942, 46)

In [ ]:
# Step 6: Keep only binding assays (assay_type == 'B' and 'F') for consistency

df_clean = df_clean[df_clean['assay_type'].isin(['B', 'F'])]
print(f"After filtering for assay type B or F: {df_clean.shape[0]} rows")

After filtering for assay type B or F: 942 rows


In [ ]:
# Step 7: Focus only on human protein targets (Homo sapiens)
df_clean = df_clean[df_clean['target_organism'].str.lower() == 'homo sapiens']

In [ ]:
df_clean.shape

(942, 46)

In [ ]:
# Step 8: Normalize units to nM (if in µM, convert to nM)
def normalize_units(row):
    if row['standard_units'] == 'µM':
        return float(row['standard_value']) * 1000  # µM to nM
    elif row['standard_units'] == 'nM':
        return float(row['standard_value'])
    else:
        return np.nan

In [ ]:
df_clean['IC50_nM'] = df_clean.apply(normalize_units, axis=1)
df_clean = df_clean.dropna(subset=['IC50_nM'])
print(f"After converting units and dropping invalid: {df_clean.shape[0]} rows")



After converting units and dropping invalid: 942 rows


In [ ]:
# Step 9: Remove rows where IC50_nM is missing
df_clean = df_clean.dropna(subset=['IC50_nM'])

In [ ]:
df_clean.shape

(942, 47)

In [ ]:
# Step 10: Remove IC50 values outside reasonable bounds (e.g., 0.1 nM to 100,000 nM)
# - IC50 < 0.1 nM is extremely rare and may reflect measurement error or unit mislabeling.
# - IC50 > 100,000 nM (100 µM) typically indicates weak or inactive compounds,
#   which are not useful for model training in drug discovery.
# This range helps retain biologically relevant and assay-reliable compounds.

df_clean = df_clean[(df_clean['IC50_nM'] >= 0.1) & (df_clean['IC50_nM'] <= 100000)]
print(f"After removing out-of-bound IC50 values: {df_clean.shape[0]} rows")



After removing out-of-bound IC50 values: 941 rows


In [ ]:
# Step 11: Calculate pIC50
df_clean['pIC50'] = -np.log10(df_clean['IC50_nM'] * 1e-9)
print("pIC50 values calculated.")

pIC50 values calculated.


In [ ]:
# Step 12: Standardize units by keeping only entries with IC50 reported in 'nM'
df_clean = df_clean[df_clean['standard_units'].str.lower() == 'nm']

In [ ]:
df_clean.shape

(941, 48)

In [ ]:
# Step 13: Reset index and inspect the cleaned DataFrame
df_clean = df_clean.reset_index(drop=True)

In [ ]:
# Step 14: Show first few rows of cleaned data
print("Preview of cleaned dataset:")
print(df_clean.head())

Preview of cleaned dataset:
  action_type activity_comment  activity_id activity_properties  \
0         NaN              NaN       103650                  []   
1         NaN              NaN       103652                  []   
2         NaN              NaN       884696                  []   
3         NaN              NaN       949636                  []   
4         NaN              NaN       949659                  []   

  assay_chembl_id                                  assay_description  \
0    CHEMBL620798  Antagonistic activity towards 5-hydroxytryptam...   
1    CHEMBL620798  Antagonistic activity towards 5-hydroxytryptam...   
2    CHEMBL618092  Inhibition against human 5-hydroxytryptamine 6...   
3    CHEMBL618090  Inhibition of human 5-hydroxytryptamine 6 rece...   
4    CHEMBL618090  Inhibition of human 5-hydroxytryptamine 6 rece...   

  assay_type  assay_variant_accession  assay_variant_mutation bao_endpoint  \
0          F                      NaN                     

In [ ]:
# Save the cleaned dataset for modeling
df_clean.to_csv("/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/processed-data/cleaned-original-data/cleaned_5-HT6_ic50.csv", index=False)
print("Cleaned dataset saved as 'cleaned_activity_data.csv'")

Cleaned dataset saved as 'cleaned_activity_data.csv'
